In [22]:
# PART 1: LOAD AND INSPECT THE DATASET

# Import the necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv("classification_escalation.csv")

# 1. Display dataset shape
print("Dataset Shape:")
print(df.shape)

# 2. Display the first 5 rows
print("\nFirst 5 Rows:")
display(df.head())

# 3. Display dataset information
print("\nDataset Information:")
df.info()

# 4. Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# 5. Check target prevalence
print("\nTarget Prevalence:")
print(df["escalated"].value_counts(normalize=True).rename("proportion"))

Dataset Shape:
(900, 6)

First 5 Rows:


,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,9,7.7,low,web,21.6,0
1,2,6.1,high,web,21.8,0
2,1,2.5,medium,phone,28.8,0
3,7,5.3,low,email,52.5,0
4,4,7.1,medium,email,14.7,0



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   complexity_score        900 non-null    int64  
 1   team_experience_years   869 non-null    float64
 2   priority                882 non-null    str    
 3   channel                 900 non-null    str    
 4   customer_tenure_months  878 non-null    float64
 5   escalated               900 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 42.3 KB

Missing Values:
complexity_score           0
team_experience_years     31
priority                  18
channel                    0
customer_tenure_months    22
escalated                  0
dtype: int64

Target Prevalence:
escalated
0    0.793333
1    0.206667
Name: proportion, dtype: float64


In [23]:

# PART 2: SPLIT THE DATA INTO TRAIN AND TEST

# Separate the features from the target
X = df.drop("escalated", axis=1)
y = df["escalated"]

# Split the dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Display the sizes
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training set shape: (675, 5)
Testing set shape: (225, 5)

Training target distribution:
escalated
0    0.792593
1    0.207407
Name: proportion, dtype: float64

Testing target distribution:
escalated
0    0.795556
1    0.204444
Name: proportion, dtype: float64


In [24]:
# PART 3: PREPROCESSING

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Identify numeric and categorical columns
numeric_features = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months"
]

categorical_features = [
    "priority",
    "channel"
]

# Numeric preprocessing pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipelines created successfully.")

Preprocessing pipelines created successfully.


In [25]:
# PART 4: MAJORITY-CLASS BASELINE

from sklearn.dummy import DummyClassifier

# Create the baseline model
baseline_model = DummyClassifier(strategy="most_frequent")

# Train the baseline model
baseline_model.fit(X_train, y_train)

# Make predictions
baseline_predictions = baseline_model.predict(X_test)

print("Baseline model trained successfully.")

Baseline model trained successfully.


In [26]:
# PART 4.2: EVALUATE BASELINE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Calculate evaluation metrics
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

baseline_precision = precision_score(
    y_test, baseline_predictions, zero_division=0
)

baseline_recall = recall_score(
    y_test, baseline_predictions, zero_division=0
)

baseline_f1 = f1_score(
    y_test, baseline_predictions, zero_division=0
)

# Display results
print("Baseline Model Results")
print("--------------------")
print("Accuracy:", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall:", baseline_recall)
print("F1 Score:", baseline_f1)

Baseline Model Results
--------------------
Accuracy: 0.7955555555555556
Precision: 0.0
Recall: 0.0
F1 Score: 0.0


In [27]:
# PART 5: LOGISTIC REGRESSION

from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
logistic_model.fit(X_train, y_train)

# Make predictions
logistic_predictions = logistic_model.predict(X_test)

# Get predicted probabilities for class 1
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [28]:
# PART 6: LINEAR DISCRIMINANT ANALYSIS

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Create the LDA pipeline
lda_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearDiscriminantAnalysis())
])

# Train the model
lda_model.fit(X_train, y_train)

# Make predictions
lda_predictions = lda_model.predict(X_test)

# Get probabilities for class 1
lda_probabilities = lda_model.predict_proba(X_test)[:, 1]

print("LDA trained successfully.")

LDA trained successfully.


In [29]:
# PART 7: K-NEAREST NEIGHBORS (K=15)

from sklearn.neighbors import KNeighborsClassifier

# Create the k-NN pipeline
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", KNeighborsClassifier(n_neighbors=15))
])

# Train the model
knn_model.fit(X_train, y_train)

# Make predictions
knn_predictions = knn_model.predict(X_test)

# Get probabilities for class 1
knn_probabilities = knn_model.predict_proba(X_test)[:, 1]

print("k-NN trained successfully.")

k-NN trained successfully.


In [30]:
# PART 8: MODEL COMPARISON

from sklearn.metrics import roc_auc_score

# Create a function to calculate metrics
def evaluate_model(y_true, predictions, probabilities=None):

    accuracy = accuracy_score(y_true, predictions)

    precision = precision_score(
        y_true, predictions, zero_division=0
    )

    recall = recall_score(
        y_true, predictions, zero_division=0
    )

    f1 = f1_score(
        y_true, predictions, zero_division=0
    )

    # Calculate ROC-AUC if probabilities are available
    if probabilities is not None:
        roc_auc = roc_auc_score(y_true, probabilities)
    else:
        roc_auc = np.nan

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc
    }


# Evaluate the baseline
baseline_results = evaluate_model(
    y_test, baseline_predictions
)

# Evaluate Logistic Regression
logistic_results = evaluate_model(
    y_test, logistic_predictions, logistic_probabilities
)

# Evaluate LDA
lda_results = evaluate_model(
    y_test, lda_predictions, lda_probabilities
)

# Evaluate k-NN
knn_results = evaluate_model(
    y_test, knn_predictions, knn_probabilities
)


# Combine results
comparison_df = pd.DataFrame(
    {
        "Majority Baseline": baseline_results,
        "Logistic Regression": logistic_results,
        "LDA": lda_results,
        "k-NN (k=15)": knn_results
    }
).T


# Display results
print("Model Comparison:")
display(comparison_df.round(4))

Model Comparison:


,Accuracy,Precision,Recall,F1,ROC-AUC
Majority Baseline,0.7956,0.00,0.0000,0.0000,NaN
Logistic Regression,0.8222,0.75,0.1957,0.3103,0.8111
LDA,0.8222,0.75,0.1957,0.3103,0.8090
k-NN (k=15),0.7956,0.50,0.0870,0.1481,0.7046
